# 01 — Data Loading & Enrichment

Load raw SIH, CNES, and SIM parquets. Filter to J96 (respiratory failure). Enrich with hospital characteristics, ICU bed inventory, equipment, staffing, and comorbidity extraction. Save clean datasets for all downstream notebooks.

**Inputs:** Raw parquets from `data/sih/`, `data/cnes/`, `data/sim/`

**Outputs:** Processed parquets in `outputs/data/`

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from pathlib import Path
from shared import (
    RAW_SIH_DIR, RAW_CNES_DIR, RAW_SIM_DIR, DATA_DIR,
    SIH_COLS, SECONDARY_DIAG_COLS, PRIMARY_ICD,
    FACILITY_TYPE_MAP, COMORBIDITY_GROUPS, COVID_ERAS,
    classify_legal_nature, classify_covid_era, extract_comorbidities,
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"SIH dir: {RAW_SIH_DIR} — exists: {RAW_SIH_DIR.exists()}")
print(f"CNES dir: {RAW_CNES_DIR} — exists: {RAW_CNES_DIR.exists()}")
print(f"SIM dir: {RAW_SIM_DIR} — exists: {RAW_SIM_DIR.exists()}")
print(f"Output dir: {DATA_DIR}")

## 1. Load SIH — Filter to J96 (Respiratory Failure)

In [ ]:
sih_files = sorted(RAW_SIH_DIR.glob("RDSP*.parquet"))
print(f"Found {len(sih_files)} SIH files")

frames = []
for f in sih_files:
    df = pd.read_parquet(f)
    available = [c for c in SIH_COLS if c in df.columns]
    df = df[available]
    mask = df["DIAG_PRINC"].astype(str).str.startswith(PRIMARY_ICD)
    frames.append(df[mask])
    
resp = pd.concat(frames, ignore_index=True)
print(f"Total J96 admissions: {len(resp):,}")
print(f"Date range: {resp['DT_INTER'].min()} to {resp['DT_INTER'].max()}")

## 2. Type Conversions & Derived Columns

In [ ]:
resp["DT_INTER"] = pd.to_datetime(resp["DT_INTER"], format="%Y%m%d", errors="coerce")
resp["DT_SAIDA"] = pd.to_datetime(resp["DT_SAIDA"], format="%Y%m%d", errors="coerce")
resp["DIAS_PERM"] = pd.to_numeric(resp["DIAS_PERM"], errors="coerce").fillna(0).astype(int)
resp["VAL_TOT"] = pd.to_numeric(resp["VAL_TOT"], errors="coerce").fillna(0)
resp["MORTE"] = pd.to_numeric(resp["MORTE"], errors="coerce").fillna(0).astype(int)
resp["IDADE"] = pd.to_numeric(resp["IDADE"], errors="coerce").fillna(0).astype(int)
resp["year"] = resp["DT_INTER"].dt.year
resp["month"] = resp["DT_INTER"].dt.month

# ICU markers
resp["icu_used"] = (resp["MARCA_UTI"].astype(str).str.strip() != "0").astype(int) if "MARCA_UTI" in resp.columns else 0
resp["icu_days"] = pd.to_numeric(resp.get("UTI_MES_TO", 0), errors="coerce").fillna(0).astype(int)

# COVID era
resp["covid_era"] = resp["DT_INTER"].apply(classify_covid_era)

# J96 subtype
resp["j96_subtype"] = resp["DIAG_PRINC"].astype(str).str[:5]

print(f"Years: {sorted(resp['year'].dropna().unique())}")
print(f"Overall mortality: {resp['MORTE'].mean():.1%}")
print(f"ICU usage rate: {resp['icu_used'].mean():.1%}")

## 3. Extract Comorbidities from Secondary Diagnoses

In [ ]:
available_diag_cols = [c for c in SECONDARY_DIAG_COLS if c in resp.columns]
print(f"Secondary diagnosis columns available: {available_diag_cols}")

comorbidity_records = resp.apply(
    lambda row: extract_comorbidities(row, available_diag_cols), axis=1
)
comorbidity_df = pd.DataFrame(comorbidity_records.tolist(), index=resp.index)
resp = pd.concat([resp, comorbidity_df], axis=1)

print("\nComorbidity prevalence:")
for col in [c for c in comorbidity_df.columns if c != "comorbidity_count"]:
    print(f"  {col}: {resp[col].mean():.1%}")
print(f"\nMean comorbidity count: {resp['comorbidity_count'].mean():.2f}")

## 4. Save Respiratory Failure SIH Dataset

In [ ]:
resp.to_parquet(DATA_DIR / "resp_failure_sih.parquet", index=False)
print(f"Saved resp_failure_sih.parquet: {len(resp):,} rows, {resp.shape[1]} columns")

## 5. Load Related Conditions (J18, J80, J44)

In [ ]:
related_icds = ["J18", "J80", "J44"]
related_cols = ["DIAG_PRINC", "DIAS_PERM", "CNES", "MUNIC_RES", "MUNIC_MOV",
                "CAR_INT", "IDADE", "SEXO", "VAL_TOT", "MORTE", "DT_INTER",
                "MARCA_UTI", "UTI_MES_TO", "ESPEC"]

related_frames = []
for f in sih_files:
    df = pd.read_parquet(f)
    available = [c for c in related_cols if c in df.columns]
    df = df[available]
    mask = df["DIAG_PRINC"].astype(str).str[:3].isin(related_icds)
    related_frames.append(df[mask])

related = pd.concat(related_frames, ignore_index=True)
related["DT_INTER"] = pd.to_datetime(related["DT_INTER"], format="%Y%m%d", errors="coerce")
related["year"] = related["DT_INTER"].dt.year
related["MORTE"] = pd.to_numeric(related["MORTE"], errors="coerce").fillna(0).astype(int)
related["icd3"] = related["DIAG_PRINC"].astype(str).str[:3]

related.to_parquet(DATA_DIR / "related_conditions.parquet", index=False)
print(f"Saved related_conditions.parquet: {len(related):,} rows")
print(related.groupby("icd3").size())

## 6. Load CNES — Facility Registry

In [ ]:
# Load most recent 12 months of CNES establishment data
cnes_files = sorted(RAW_CNES_DIR.glob("STSP*.parquet"))
print(f"Found {len(cnes_files)} CNES files")

# Use last 12 months for most current snapshot
recent_cnes_files = cnes_files[-12:] if len(cnes_files) > 12 else cnes_files
cnes_frames = [pd.read_parquet(f) for f in recent_cnes_files]
cnes = pd.concat(cnes_frames, ignore_index=True)

# Deduplicate: keep most recent record per CNES
cnes["CNES"] = cnes["CNES"].astype(str).str.strip()
cnes = cnes.drop_duplicates(subset=["CNES"], keep="last")

# Filter to hospitals that treated J96 patients
j96_hospitals = resp["CNES"].astype(str).str.strip().unique()
cnes_j96 = cnes[cnes["CNES"].isin(j96_hospitals)].copy()

cnes_j96.to_parquet(DATA_DIR / "resp_failure_cnes.parquet", index=False)
print(f"Saved resp_failure_cnes.parquet: {len(cnes_j96):,} hospitals")

## 7. Load CNES LT — ICU Bed Inventory

In [ ]:
lt_files = sorted(RAW_CNES_DIR.glob("LTSP*.parquet"))
print(f"Found {len(lt_files)} CNES LT (beds) files")

if lt_files:
    lt = pd.read_parquet(lt_files[-1])  # Most recent snapshot
    lt["CNES"] = lt["CNES"].astype(str).str.strip()
    
    # Aggregate bed counts per hospital
    # TP_LEITO codes: 74=ICU Adult, 75=ICU Pediatric, 76=ICU Neonatal, etc.
    # QT_SUS = beds available to SUS
    lt_agg = lt.groupby("CNES").agg(
        total_beds=("QT_EXIST", "sum"),
        total_beds_sus=("QT_SUS", "sum"),
    ).reset_index()
    
    # ICU beds specifically
    icu_mask = lt["TP_LEITO"].astype(str).str.startswith("7")
    icu_beds = lt[icu_mask].groupby("CNES").agg(
        icu_beds_total=("QT_EXIST", "sum"),
        icu_beds_sus=("QT_SUS", "sum"),
    ).reset_index()
    
    icu_inventory = lt_agg.merge(icu_beds, on="CNES", how="left").fillna(0)
    icu_inventory.to_parquet(DATA_DIR / "hospital_icu_beds.parquet", index=False)
    print(f"Saved hospital_icu_beds.parquet: {len(icu_inventory):,} hospitals")
else:
    print("WARNING: No CNES LT files found. Download with: sus-pipeline download CNES --years 2024-2025 --uf SP --group LT")

## 8. Hospital Classification Tags

In [ ]:
# Build hospital tags from CNES + SIH data
tags = pd.DataFrame({"CNES": j96_hospitals})

# Facility type from CNES
if len(cnes_j96) > 0 and "TP_UNID" in cnes_j96.columns:
    type_map = cnes_j96.set_index("CNES")["TP_UNID"].astype(str).map(FACILITY_TYPE_MAP).to_dict()
    tags["broad_type"] = tags["CNES"].map(type_map).fillna("other")
else:
    tags["broad_type"] = "unknown"

# Legal nature from CNES
if len(cnes_j96) > 0 and "NAT_JUR" in cnes_j96.columns:
    nat_map = cnes_j96.set_index("CNES")["NAT_JUR"].apply(classify_legal_nature).to_dict()
    tags["nat_jur_category"] = tags["CNES"].map(nat_map).fillna("unknown")
else:
    tags["nat_jur_category"] = "unknown"

# Admission profile from SIH
hosp_profile = resp.groupby("CNES").agg(
    total=pd.NamedAgg(column="MORTE", aggfunc="count"),
    emergency_rate=pd.NamedAgg(column="CAR_INT", aggfunc=lambda x: (x.astype(str) == "02").mean()),
).reset_index()
hosp_profile["CNES"] = hosp_profile["CNES"].astype(str).str.strip()
hosp_profile["admission_profile"] = np.where(
    hosp_profile["emergency_rate"] > 0.6, "emergency_dominant",
    np.where(hosp_profile["emergency_rate"] < 0.4, "elective_dominant", "mixed")
)
tags = tags.merge(hosp_profile[["CNES", "admission_profile"]], on="CNES", how="left")

# ICU capability level
if "icu_inventory" in dir() and len(icu_inventory) > 0:
    icu_map = icu_inventory.set_index("CNES")["icu_beds_sus"]
    tags["icu_beds"] = tags["CNES"].map(icu_map).fillna(0)
    tags["icu_capacity_level"] = pd.cut(
        tags["icu_beds"], bins=[-1, 0, 10, 30, 999],
        labels=["no_icu", "small_icu", "medium_icu", "large_icu"]
    )
else:
    tags["icu_beds"] = 0
    tags["icu_capacity_level"] = "unknown"

# Comparability group
tags["comparability_group"] = (
    tags["broad_type"].astype(str) + "__" +
    tags["admission_profile"].astype(str) + "__" +
    tags["icu_capacity_level"].astype(str)
)

tags.to_parquet(DATA_DIR / "hospital_tags.parquet", index=False)
print(f"Saved hospital_tags.parquet: {len(tags):,} hospitals")
print(f"\nFacility types:\n{tags['broad_type'].value_counts()}")
print(f"\nICU capacity levels:\n{tags['icu_capacity_level'].value_counts()}")

## 9. Load SIM — Respiratory Failure Mortality Records

In [ ]:
sim_files = sorted(RAW_SIM_DIR.glob("DOSP*.parquet"))
print(f"Found {len(sim_files)} SIM files")

if sim_files:
    sim_frames = []
    for f in sim_files:
        df = pd.read_parquet(f)
        mask = df["CAUSABAS"].astype(str).str.startswith(PRIMARY_ICD)
        sim_frames.append(df[mask])
    
    sim_resp = pd.concat(sim_frames, ignore_index=True)
    sim_resp.to_parquet(DATA_DIR / "sim_respiratory.parquet", index=False)
    print(f"Saved sim_respiratory.parquet: {len(sim_resp):,} death records with J96 as cause")
else:
    print("WARNING: No SIM files found. Download with: sus-pipeline download SIM --years 2016-2024 --uf SP")

## 10. Summary

In [ ]:
print("=" * 60)
print("DATA LOADING COMPLETE")
print("=" * 60)
print(f"\nJ96 admissions: {len(resp):,}")
print(f"Date range: {resp['DT_INTER'].min().date()} to {resp['DT_INTER'].max().date()}")
print(f"Years covered: {sorted(resp['year'].dropna().unique().astype(int))}")
print(f"Unique hospitals: {resp['CNES'].nunique():,}")
print(f"Unique municipalities: {resp['MUNIC_MOV'].nunique():,}")
print(f"\nMortality rate: {resp['MORTE'].mean():.1%}")
print(f"ICU usage rate: {resp['icu_used'].mean():.1%}")
print(f"Mean LOS: {resp['DIAS_PERM'].mean():.1f} days")
print(f"Total cost: R$ {resp['VAL_TOT'].sum():,.0f}")
print(f"\nMortality by era:")
for era in ["pre_covid", "covid_acute", "post_covid_early", "post_covid_late"]:
    subset = resp[resp["covid_era"] == era]
    if len(subset) > 0:
        print(f"  {era}: {subset['MORTE'].mean():.1%} ({len(subset):,} admissions)")

print(f"\nOutputs saved to: {DATA_DIR.resolve()}")
for f in sorted(DATA_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")